# RAG From Scratch — Open-Source / Hugging Face Edition


#### What changes vs. the OpenAI version, and what doesn't:


| Role | OpenAI version | This notebook |
|---|---|---|
| Embedding model | `OpenAIEmbeddings` (API call, costs money) | `HuggingFaceEmbeddings` — runs **locally** via `sentence-transformers`, free |
| Chat / generation model | `ChatOpenAI` (API call, costs money) | `HuggingFacePipeline` + `ChatHuggingFace` — runs **locally** via `transformers`, free |
| Tokenizer | `tiktoken` (OpenAI-specific) | the model's own Hugging Face tokenizer (`AutoTokenizer`) |


The Vector store with `Chroma`, Text splitting with `RecursiveCharacterTextSplitter`, the  Document loading with `WebBaseLoader` all remain **unchanged** 

LangChain's abstractions (`Runnable`, `.invoke()`, the `\|` operator) are provider-agnostic. Swapping OpenAI for Hugging Face only touches *two lines* — how the embedding model and the chat model are constructed. Everything downstream — chunking, storing, retrieving, prompting, chaining — is identical code. That's the whole point of the abstraction

**Trade-off to be upfront about:** open models you can run on a laptop/desktop GPU are noticeably weaker at instruction-following than GPT-4o-mini. Retrieval quality will look basically identical (that part doesn't depend on OpenAI at all); generation quality will be visibly rougher. That's fine — the goal here is understanding *how RAG works*, not building a production chatbot. The first run will **download both models** from the Hugging Face Hub (a few hundred MB for the embedding model, a couple GB for the LLM) — that only happens once, they're cached locally afterward.

####  Environment setup

New packages needed on top of the OpenAI notebook's dependencies:

| Package | Role |
|---|---|
| `langchain-huggingface` | LangChain's official Hugging Face integration — `HuggingFaceEmbeddings`, `HuggingFacePipeline`, `ChatHuggingFace` |
| `sentence-transformers` | the library that actually runs embedding models locally (wrapped by `HuggingFaceEmbeddings`) |
| `transformers` | Hugging Face's model-loading/inference library (wrapped by `HuggingFacePipeline`) |
| `torch` | the tensor/deep-learning library everything above runs on |

All five are already declared in this project's `pyproject.toml`. No API key is required for any of this — models download once from the Hugging Face Hub (public, anonymous access is fine for the models used here) and then run locally.

### A note on GPU vs. CPU

The code below auto-detects whether PyTorch can see a CUDA GPU and uses it automatically; if not, it falls back to CPU (slower, but the small models used here are still usable on CPU — generation just takes longer, seconds rather than instant).


In [1]:
from dotenv import load_dotenv
load_dotenv()



import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DEVICE = 0 if torch.cuda.is_available() else -1  # transformers convention: GPU index, or -1 for CPU

CUDA available: True
GPU: NVIDIA GeForce RTX 4080 SUPER


### Models used in this notebook

Both models below are public and **ungated** on the Hugging Face Hub — no login, no license click-through, no token needed to download them.

- **`sentence-transformers/all-MiniLM-L6-v2`** for embeddings — a small (22M parameter), fast, widely-used sentence embedding model. Not as strong as OpenAI's `text-embedding-3-small`, but a very standard, well-understood baseline for exactly this kind of tutorial.
- **`Qwen/Qwen2.5-1.5B-Instruct`** for generation — a 1.5-billion-parameter instruction-tuned chat model, small enough to run comfortably on CPU (and fast on any modern GPU). If you have a decent GPU and want noticeably better answers, swap this for a bigger sibling like `Qwen/Qwen2.5-7B-Instruct` — the code doesn't change, just the string.

In [2]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # 384-dimensional
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"                    # swap for a bigger Qwen/Llama/Mistral variant if your hardware allows

---
# Part 1 — Quickstart: the full pipeline, end to end

Same shape as the OpenAI notebook: run the whole thing once, then unpack each stage below. Source document is the same too — Lilian Weng's blog post on LLM-powered autonomous agents — so you can compare answer quality against the OpenAI version if you have both running.



In [3]:
import bs4

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline


from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough


def format_docs_function(docs):
    return "\n\n".join(doc.page_content for doc in docs)

#### INDEXING ####

# 1. Load: fetch the page, keep only the title/header/body HTML
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

# 2. Split: break the article into small, overlapping chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# 3. Embed + store: turn every chunk into a vector (locally, via sentence-transformers), index it in Chroma
# 'HuggingFaceEmbeddings' will just load the specified embedding model from HuggingFace and call it on each chunk. The model is cached locally, so the first run may take a while.
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

# Chroma.from_documents does three things foe eacj chunk in the list `splits`:
#   (a) calls embeddings.embed_documents(...) to turn splits[i].page_content into a 384-dim vector
#   (b) generates an id for the chunk and stores the row {id, vector, page_content, metadata} in Chroma's on-disk store (so the vector can be traced back to its original text)
#   (c) adds the vector to an HNSW approximate-nearest-neighbor index for fast future similarity search
# The `embeddings` object is kept by reference so the *same* model embeds future queries too.
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

# .as_retriever() wraps the vectorstore in a LangChain Runnable. Calling .invoke(query) on it will:
#   1. embed the query string with the same `embeddings` model used above
#   2. run nearest-neighbor search over the HNSW index to find the closest stored vectors
#   3. use the ids of those vectors to look up and return the original Document objects (text + metadata)
retriever = vectorstore.as_retriever()

#### RETRIEVAL AND GENERATION ####


# 4. Load the local LLM - a local transformers text-generation pipeline
llm_pipeline = HuggingFacePipeline.from_model_id(model_id=LLM_MODEL,
    task="text-generation",
    device=DEVICE,
    pipeline_kwargs={
        "max_new_tokens": 512,
        "do_sample": False,        # deterministic output, same intent as ChatOpenAI(temperature=0)
        "return_full_text": False,  # return only the new text, not prompt+completion
    },
)

llm = ChatHuggingFace(llm=llm_pipeline)


# define a template prompt
prompt = ChatPromptTemplate.from_template(
        "You are an assistant for question-answering tasks. Use the following "
        "pieces of retrieved context to answer the question. If you don't know "
        "the answer, just say that you don't know. Use three sentences maximum "
        "and keep the answer concise.\n"
        "Question: {question}\nContext: {context}\nAnswer:"
)


# 5. Chain: wire everything into one Runnable with the `|` operator.
# `|` is LangChain's pipe operator (like a Unix pipe) — the output of the step on the left becomes the input of the step on the right. Reading top to bottom:
#
#    a) {"context": retriever | format_docs, "question": RunnablePassthrough()} :
#       A dict of Runnables is itself a Runnable ("RunnableParallel"). Given the input question string, it runs both entries and returns a dict of results:
#         - "context"  -> retriever.invoke(question) fetches the top-k similar Documents, then format_docs_function joins their text into one string
#         - "question" -> RunnablePassthrough() just passes the question through unchanged
#       Result: {"context": "<retrieved chunk text>", "question": "<the question>"}
#
#    b) | prompt              -> fills {context}/{question} into the prompt template
#    c) | llm                 -> sends the filled prompt to the model, gets an AIMessage back
#    d) | StrOutputParser()   -> unwraps AIMessage.content into a plain string
rag_chain = (
    {"context": retriever | format_docs_function, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 8. Ask a question
response = rag_chain.invoke("What is the idea of Reflexion  ?")
print(response)

C:\Users\Amine\AppData\Local\Temp\ipykernel_36264\386951472.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
c:\Users\Amine\PycharmProjects\rag-from-scratch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 8307.16it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `g

Reflexion is a framework designed to enhance the reasoning abilities of agents through dynamic memory management and self-reflection. It equips agents with the capability to learn from their experiences and adapt their strategies accordingly. The reflexive process involves analyzing the outcomes of previous actions and using them to inform future decisions. This approach aims to reduce inefficiencies and avoid hallucinations in decision-making processes.


In [9]:
print(len(splits))
print(splits[:3])

63
[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and r

That's the whole pipeline, running on entirely free, local models. Now let's open up each stage — same order as before: indexing, retrieval, generation.

---
# Part 2 — Indexing

Indexing turns raw documents into something searchable by *meaning*.

It has 4 sub-steps: **load → split → embed → store**. As before, we'll build intuition on two tiny hand-written sentences first, then apply it to the full article.

## 2.1 Tokens

Every model — embedding models and LLMs alike — operates on **tokens**, not raw characters or whole words. In the OpenAI notebook we used `tiktoken` to count tokens; here, there's an even more direct way to see it: every Hugging Face model ships with **its own tokenizer**, and we can load it and use it directly.

**Note** : the embedding model and the LLM in this notebook are two *different* models, so they have two *different* tokenizers, with different vocabularies and different token boundaries for the same text. LangChain handles this correctly for you automatically (each model applies its own tokenizer internally), but it's worth seeing directly — it's a good example of why "the same text" isn't a fixed number of tokens; it depends on which model is doing the counting.

In [4]:
from transformers import AutoTokenizer

question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

embedding_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

for label, tok in [("embedding model", embedding_tokenizer), ("LLM", llm_tokenizer)]:
    n_q = len(tok.encode(question))
    n_d = len(tok.encode(document))
    print(f"{label:16s} ({tok.name_or_path}):")
    print(f"  {n_q} tokens -> {question!r}")
    print(f"  {n_d} tokens -> {document!r}")

embedding model  (sentence-transformers/all-MiniLM-L6-v2):
  10 tokens -> 'What kinds of pets do I like?'
  9 tokens -> 'My favorite pet is a cat.'
LLM              (Qwen/Qwen2.5-1.5B-Instruct):
  8 tokens -> 'What kinds of pets do I like?'
  7 tokens -> 'My favorite pet is a cat.'


## 2.2 Embeddings

An **embedding model** maps text to a fixed-length vector such that semantically similar text ends up close together in that vector space. Under the hood, `sentence-transformers` models like `all-MiniLM-L6-v2` work in two steps: run the text through a transformer encoder to get one vector *per token*, then **mean-pool** those token vectors into a single fixed-length sentence vector.

The same two rules from the OpenAI notebook still apply, unchanged:

- **Vector size is fixed per model.** `all-MiniLM-L6-v2` always returns 384 numbers (that's what the "384" in most sentence-transformers benchmark tables refers to) — smaller than OpenAI's 1536/3072, which is exactly the accuracy/speed/storage trade-off that makes this model small and fast enough to run on a CPU in a tutorial.
- **You must embed the query with the exact same model used to embed your documents.** `HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)` is the one instance used for both indexing (`Chroma.from_documents`) and every query the retriever embeds later — same as before, just a different model doing the embedding.

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embd = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

query_result = embd.embed_query(question)
document_result = embd.embed_query(document)
unrelated = embd.embed_query("The stock market fell sharply on Tuesday.")

print(f"Embedding model      : {EMBEDDING_MODEL}")
print(f"Vector length (dims)  : {len(query_result)}")
print(f"First 5 values of the question vector: {query_result[:5]}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10430.32it/s]


Embedding model      : sentence-transformers/all-MiniLM-L6-v2
Vector length (dims)  : 384
First 5 values of the question vector: [0.030804593116044998, -0.054369471967220306, 0.1035204827785492, 0.04905468970537186, -0.10118032991886139]


## 2.3 Similarity

Same cosine similarity computation as before — this part of RAG has nothing to do with which company made the embedding model, it's just linear algebra:

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\lVert A \rVert \, \lVert B \rVert}$$

One practical difference worth calling out: smaller, older-architecture embedding models like MiniLM tend to produce cosine similarities in a narrower, lower band than OpenAI's newest embedding models — don't be surprised if even a strongly related pair scores, say, 0.5–0.6 rather than 0.9+. What matters for retrieval is the **relative ranking** (related text scores higher than unrelated text), not the absolute number.

In [6]:
import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

print("question  vs. related document  :", cosine_similarity(query_result, document_result))
print("question  vs. unrelated sentence:", cosine_similarity(query_result, unrelated))
print("document  vs. itself            :", cosine_similarity(document_result, document_result))

question  vs. related document  : 0.5782656253169495
question  vs. unrelated sentence: -0.016569566516781438
document  vs. itself            : 1.0


Related still clearly outscores unrelated, and a vector against itself is still exactly 1.0. This is the computation a vector store runs internally on every query — brute-force here for two vectors, an approximate-nearest-neighbor index (HNSW) at scale.

## 2.4 Loading documents

Unchanged from the OpenAI notebook — document loading doesn't involve OpenAI or Hugging Face at all, it's just fetching and parsing raw source content into LangChain's common `Document` object (`page_content` + `metadata`).

In [7]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

print(f"Loaded {len(blog_docs)} document(s)")
print(f"Metadata: {blog_docs[0].metadata}")
print(f"Length: {len(blog_docs[0].page_content)} characters / {len(llm_tokenizer.encode(blog_docs[0].page_content))} LLM tokens")

Loaded 1 document(s)
Metadata: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}
Length: 43047 characters / 9629 LLM tokens


## 2.5 Splitting

Same reasoning as before: one embedding vector per giant article blurs everything together, and models have input-size limits anyway, so we split into small overlapping chunks.

One nice option available with an open tokenizer: instead of `.from_tiktoken_encoder(...)`, LangChain also provides **`.from_huggingface_tokenizer(...)`**, which counts `chunk_size` using *this exact embedding model's own tokenizer* — so the chunk boundaries line up with how the embedding model itself will actually consume the text, rather than an OpenAI tokenizer that has nothing to do with the model doing the embedding.

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    embedding_tokenizer,
    chunk_size=300,
    chunk_overlap=50,
)

splits = text_splitter.split_documents(blog_docs)

print(f"{len(blog_docs)} document(s) -> {len(splits)} chunks")
print(f"Chunk 0 token count (embedding model's tokenizer): {len(embedding_tokenizer.encode(splits[0].page_content))}")
print(f"Chunk 0 preview: {splits[0].page_content[:200]}...")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (584 > 512). Running this sequence through the model will result in indexing errors


1 document(s) -> 52 chunks
Chunk 0 token count (embedding model's tokenizer): 276
Chunk 0 preview: LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool con...


## 2.6 Storing in a vector store

Unchanged — `Chroma.from_documents` still does the same three things (embed every chunk, store `{vector, page_content, metadata}`, build an HNSW index for fast search), it just calls `HuggingFaceEmbeddings` instead of `OpenAIEmbeddings` to get each vector. Chroma itself has no idea — and doesn't care — which company made the embedding model; it just stores numbers.

In [9]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL),
)

print(f"Indexed {vectorstore._collection.count()} chunks")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12106.30it/s]


Indexed 115 chunks


### 2.6.1 Peeking inside the vector store

`vectorstore` doesn't just hold vectors — every stored row is `{id, embedding, document (the original chunk text), metadata}`. The cell below pulls a few rows straight out of Chroma's underlying collection and lines the vector up next to the text it came from, so the id-based mapping between "vector" and "string" is visible instead of implicit.

In [10]:
import numpy as np
import pandas as pd


def peek_vectorstore(vectorstore, n=5, n_dims=6, text_chars=90):
    """Show `n` stored rows as a table: id, text preview, vector preview, and full dim count.

    Confirms the two things Chroma actually stores per chunk: the embedding vector
    AND the original text (+ metadata), joined by a shared id.
    """
    store = vectorstore._collection.get(limit=n, include=["embeddings", "documents", "metadatas"])

    rows = []
    for id_, text, vector, meta in zip(
        store["ids"], store["documents"], store["embeddings"], store["metadatas"]
    ):
        rows.append(
            {
                "id": id_[:8] + "…",
                "text_preview": text[:text_chars] + ("…" if len(text) > text_chars else ""),
                "vector_preview": np.round(np.asarray(vector[:n_dims]), 3).tolist(),
                "dims": len(vector),
            }
        )
    return pd.DataFrame(rows)


peek_vectorstore(vectorstore, n=5)

,id,text_preview,vector_preview,dims
0,7b35bc64…,LLM Powered Autonomous Agents\n \nDate: Jun...,"[-0.007, -0.001, 0.033, 0.015, -0.013, -0.034]",384
1,0db6fa94…,Memory\n\nShort-term memory: I would consider ...,"[0.016, 0.002, -0.049, 0.022, 0.067, 0.012]",384
2,2e274144…,Component One: Planning#\nA complicated task u...,"[0.032, 0.023, 0.026, -0.046, 0.081, -0.068]",384
3,14ca4f79…,Task decomposition can be done (1) by LLM with...,"[-0.047, 0.005, 0.027, -0.049, 0.013, -0.047]",384
4,098ca69f…,Self-Reflection#\nSelf-reflection is a vital a...,"[-0.003, -0.062, 0.08, -0.002, 0.051, 0.014]",384


---
# Part 3 — Retrieval

Also unchanged in every way that matters: `retriever.invoke(question)` embeds the question with the same embedding model used at indexing time, runs a nearest-neighbor search over the stored vectors, and returns the top-`k` most similar chunks. None of that logic references OpenAI or Hugging Face specifically — retrieval is provider-agnostic once the vectors exist.

In [11]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

retrieved_docs = retriever.invoke("What is Task Decomposition?")

print(f"Retrieved {len(retrieved_docs)} document(s)")
print(retrieved_docs[0].page_content)

Retrieved 1 document(s)
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.
Tree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evaluated by a classifier (via a prompt) or majority vote.
Task decomposition can be done (1) b

Same note as before on `search_type="mmr"` (diversify results) and `search_type="similarity_score_threshold"` (drop weak matches) — both work identically regardless of which embedding model produced the vectors.

---
# Part 4 — Generation

This is the stage where swapping to Hugging Face actually changes the code, because we're now running a local text-generation model instead of calling an API. Let's build it up step by step.

In [12]:
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

## 4.1 The LLM: running an open model locally

`ChatOpenAI` was a thin wrapper around an HTTP call to OpenAI's servers. Its local equivalent needs two pieces, chained together:

1. **`HuggingFacePipeline`** — wraps a `transformers` `text-generation` pipeline: downloads `LLM_MODEL`'s weights (once, then cached), loads them onto the GPU if available (else CPU), and runs raw text-in / text-out generation.
2. **`ChatHuggingFace`** — wraps that pipeline so it exposes the same *chat-message* interface `ChatOpenAI` does (`AIMessage` in, `AIMessage` out). It does this by applying the model's own **chat template** — a Jinja template baked into the tokenizer's config that wraps your messages in whatever special tokens the model was trained on (e.g. Qwen uses ChatML-style `<|im_start|>user ... <|im_end|>` markers). This is the same reason ChatGPT-style models need a specific message format under the hood — `ChatHuggingFace` just makes that explicit and automatic instead of hidden behind OpenAI's API.

A couple of pipeline settings matter enough to call out:
- `do_sample=False` — deterministic, greedy decoding. Same intent as `ChatOpenAI(temperature=0)`.
- `return_full_text=False` — by default, `transformers`' text-generation pipeline returns the *entire* prompt plus the completion appended after it; this flag tells it to return only the newly generated text, which is what we actually want downstream.

In [13]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

llm_pipeline = HuggingFacePipeline.from_model_id(
    model_id=LLM_MODEL,
    task="text-generation",
    device=DEVICE,
    pipeline_kwargs={
        "max_new_tokens": 512,
        "do_sample": False,
        "return_full_text": False,
    },
)
llm = ChatHuggingFace(llm=llm_pipeline)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 9889.47it/s]


## 4.3 A minimal chain

Same `prompt | llm` pipe as the OpenAI notebook — because `ChatHuggingFace` implements the same `Runnable` interface `ChatOpenAI` does, this line of code is byte-for-byte identical between the two notebooks.

In [14]:
chain = prompt | llm

response = chain.invoke({"context": retrieved_docs, "question": "What is Task Decomposition?"})
print(response.content)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Task Decomposition is a method used in planning that breaks down a complex task into smaller, manageable steps. This approach helps agents understand how to tackle difficult tasks by breaking them down systematically. It enhances model performance on complex tasks by utilizing more computational resources during testing time. Task decomposition can be achieved through various techniques such as Chain of Thought (Wei et al., 2022), which guides the model to think step-by-step, transforming large tasks into multiple simpler ones, and Tree of Thoughts (Yao et al., 2023), which explores different reasoning possibilities at each step, generating multiple thoughts per step, and creating a structured tree-like representation.


`response` is still an `AIMessage`, exactly like the OpenAI version — that's the abstraction working as intended. Don't expect GPT-4o-mini-level fluency from a 1.5B model, but the mechanism is identical.

## 4.5 The full RAG chain

The same chain from the Part 1 quickstart, now fully unpacked. Compare this cell to the equivalent one in the OpenAI notebook: the only difference is that `llm` was built with `HuggingFacePipeline` + `ChatHuggingFace` instead of `ChatOpenAI`, and `retriever` embeds with `HuggingFaceEmbeddings` instead of `OpenAIEmbeddings`. The chain assembly itself — `retriever | format_docs`, `RunnablePassthrough()`, `prompt`, `llm`, `StrOutputParser()` — doesn't change at all.

In [15]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retriever = vectorstore.as_retriever()

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Task Decomposition refers to breaking down a complex task into smaller, manageable parts or sub-goals that can be addressed individually before combining them to achieve the overall objective. This approach helps simplify large problems by focusing on incremental progress rather than tackling everything at once. It enhances understanding of how models think through the process of dividing tasks into logical steps and evaluating different approaches within those steps.'

---
## Recap

| Stage | What happens | Key objects (this notebook) |
|---|---|---|
| **Indexing** | load → split → embed → store | `WebBaseLoader`, `RecursiveCharacterTextSplitter`, `HuggingFaceEmbeddings`, `Chroma` |
| **Retrieval** | embed the question with the *same* embedding model, nearest-neighbor search | `vectorstore.as_retriever()`, `.invoke(question)` |
| **Generation** | fill retrieved context + question into a prompt, run the LLM, parse the response | `ChatPromptTemplate`, `HuggingFacePipeline` + `ChatHuggingFace`, `StrOutputParser` |

### What actually changed vs. the OpenAI notebook

- `OpenAIEmbeddings(model=...)` → `HuggingFaceEmbeddings(model_name=...)` — local, free, no key
- `ChatOpenAI(model=..., temperature=0)` → `HuggingFacePipeline.from_model_id(...)` wrapped in `ChatHuggingFace(llm=...)` — local, free, no key, but needs a `return_full_text=False` setting the API-based version never had to think about
- `tiktoken.get_encoding(...)` → `AutoTokenizer.from_pretrained(model_id)` — count tokens using the *actual* model's own tokenizer
- `RecursiveCharacterTextSplitter.from_tiktoken_encoder(...)` → `.from_huggingface_tokenizer(tokenizer, ...)`

### What didn't change at all

`WebBaseLoader`, `RecursiveCharacterTextSplitter` (the class itself), `Chroma`, `vectorstore.as_retriever()`, `ChatPromptTemplate`, `langchain_classic.hub`, `RunnablePassthrough`, `StrOutputParser`, and the entire `|` chain-composition pattern. That's the real takeaway: LangChain's job is to make the *plumbing* (loading, splitting, storing, retrieving, prompting, chaining) independent of *which* embedding model or *which* LLM you plug in. Once you're comfortable with this notebook, swapping in a different embedding model, a bigger local LLM, or even a hosted-but-free option like `HuggingFaceEndpoint` (Hugging Face's hosted Inference API, using a free HF account token instead of running weights locally) is a one-line change, not a rewrite.

### What's next

The other notebooks in this repo (`rag_from_scratch_5_to_9.ipynb` onward) build advanced retrieval strategies — multi-query retrieval, RAG-fusion, query decomposition, step-back prompting — on top of this same three-stage foundation. Every one of them will work with either the OpenAI or the Hugging Face `retriever`/`llm` objects built here, since they're all just `Runnable`s underneath.